# PREPROCESSING

In [ ]:
!pip install -q nltk contractions emoji pyspellchecker Sastrawi openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.8 MB/s eta 0:00:00


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re, sys, time
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import contractions
import emoji
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm
from spellchecker import SpellChecker
from collections import Counter
import numpy as np

In [ ]:
BASE_URL = "https://pta.trunojoyo.ac.id/c_search/byprod"

In [ ]:
# Download tokenizer
nltk.download("punkt")
nltk.download("punkt_tab")

# Download stopwords
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
def get_max_page(prodi_id):
    url = f"{BASE_URL}/{prodi_id}/1"
    r = requests.get(url)
    soup = BeautifulSoup(r.content, "html.parser")

    # Cari tombol last page
    last_page = soup.select_one('ol.pagination a:contains("»")')
    if last_page and "href" in last_page.attrs:
        href = last_page["href"]
        max_page = int(href.split("/")[-1])
        return max_page

    # fallback kalau pagination tidak ada
    return 1

In [ ]:
# Tes get_max_page
print(get_max_page(23))

34


/usr/local/lib/python3.12/dist-packages/soupsieve/css_parser.py:876: FutureWarning: The pseudo class ':contains' is deprecated, ':-soup-contains' should be used moving forward.
  warnings.warn(  # noqa: B028


In [ ]:
def print_progress(prodi_id, prodi, current_page, total_pages):
    percent = (current_page / total_pages) * 100
    bar_length = 20
    filled_length = int(bar_length * current_page // total_pages)
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    sys.stdout.write(f'\r[{prodi_id}] {prodi} - Page {current_page}/{total_pages} [{bar}] {percent:.2f}%')
    sys.stdout.flush()
    if current_page == total_pages:
        sys.stdout.write('\n')

In [ ]:
def pta():
    start_time = time.time()

    data = {
        "id": [],
        "penulis": [],
        "judul": [],
        "abstrak_id": [],
        "abstrak_en": [],
        "pembimbing_pertama": [],
        "pembimbing_kedua": [],
        "prodi": []
    }

    # daftar prodi yang akan diproses
    prodi_list = [7]
    total_pages = 0
    max_pages_dict = {}

    # hitung total halaman (untuk tiap prodi yang dipilih)
    for i in prodi_list:
        max_page = get_max_page(i)
        max_pages_dict[i] = max_page
        total_pages += max_page

    # scraping data tiap prodi
    for i in prodi_list:
        max_page = max_pages_dict[i]
        for j in range(1, max_page + 1):
            url = f"{BASE_URL}/{i}/{j}"
            r = requests.get(url)
            soup = BeautifulSoup(r.content, "html.parser")
            jurnals = soup.select('li[data-cat="#luxury"]')

            isii = soup.select_one('div#begin')
            if not isii:
                continue
            prodi_full = isii.select_one('h2').text.strip()
            prodi = prodi_full.replace("Journal Jurusan ", "")

            for jurnal in jurnals:
                link_keluar = jurnal.select_one('a.gray.button')['href']

                # ambil ID dari link PTA
                id_match = re.search(r"/detail/(\d+)", link_keluar)
                pta_id = id_match.group(1) if id_match else None

                response = requests.get(link_keluar)
                soup1 = BeautifulSoup(response.content, "html.parser")
                isi = soup1.select_one('div#content_journal')

                judul = isi.select_one('a.title').text.strip()
                penulis = isi.select_one('span:contains("Penulis")').text.split(' : ')[1]
                pembimbing_pertama = isi.select_one('span:contains("Dosen Pembimbing I")').text.split(' : ')[1]
                pembimbing_kedua = isi.select_one('span:contains("Dosen Pembimbing II")').text.split(' :')[1]

                paragraf = isi.select('p[align="justify"]')
                abstrak_id = paragraf[0].get_text(strip=True) if len(paragraf) > 0 else "N/A"
                abstrak_en = paragraf[1].get_text(strip=True) if len(paragraf) > 1 else "N/A"

                data["id"].append(pta_id)
                data["penulis"].append(penulis)
                data["judul"].append(judul)
                data["abstrak_id"].append(abstrak_id)
                data["abstrak_en"].append(abstrak_en)
                data["pembimbing_pertama"].append(pembimbing_pertama)
                data["pembimbing_kedua"].append(pembimbing_kedua)
                data["prodi"].append(prodi)

            # update progress bar per prodi
            print_progress(i, prodi, j, max_page)

    # simpan ke Excel
    df = pd.DataFrame(data)
    df.to_excel("pta.xlsx", index=False, engine='openpyxl')

    # hitung durasi
    end_time = time.time()
    elapsed = int(end_time - start_time)
    jam, sisa = divmod(elapsed, 3600)
    menit, detik = divmod(sisa, 60)

    # summary
    print("\n✅ Seluruh data berhasil dikumpulkan!")
    print(f"📊 Total entri: {len(df)}")
    print(f"⏱️ Waktu eksekusi: {jam} jam {menit} menit {detik} detik")

    return None

In [ ]:
pta()

[7] Manajemen - Page 207/207 [████████████████████] 100.00%

✅ Seluruh data berhasil dikumpulkan!
📊 Total entri: 1031
⏱️ Waktu eksekusi: 0 jam 31 menit 39 detik


In [ ]:
pta_df = pd.read_excel("pta.xlsx", engine='openpyxl')
pta_df

,id,penulis,judul,abstrak_id,abstrak_en,pembimbing_pertama,pembimbing_kedua,prodi
0,80211100070,SATIYAH,PENGARUH FAKTOR-FAKTOR PELATIHAN DAN PENGEMBAN...,"ABSTRAK_x000D_\nSatiyah, Pengaruh Faktor-fakto...",ABSTRACT_x000D_\n_x000D_\nIn an effort to incr...,"Dra. Hj. S. Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST.SE,M.MT",Manajemen
1,90211200001,Faishal,ANALISIS PERSEPSI BRAND ASSOCIATION MENURUT PE...,Tujuan penelitian ini adalah untuk mengetahui ...,This study wanted to know the brand associatio...,Nurita Andriani,Yustina Chrismardani,Manajemen
2,80211100050,Wahyu Kurniawan,PENGARUH GAYA KEPEMIMPINAN DEMOKRATIK TERHADAP...,NaN,NaN,"Dr. Dra. Hj. Iriani Ismail, MM","Dra. Hj. S. Anugrahini Irawati, MM",Manajemen
3,100211200002,Muhammad Zakaria Utomo,Pengukuran Website Quality Pada Situs Sistem A...,Aplikasi nyata pemanfaatan teknologi informasi...,Academic portal system in University of Trunoj...,"Dr. Ir. Nurita Andriani, MM","Nirma Kurriwati, SP, M.Si",Manajemen
4,80211100044,Hendri Wahyudi Prayitno,PENGARUH KEPEMIMPINAN DAN KOMPENSASI TERHADAP ...,Abstrak_x000D_\nPenelitian ini menggunakan met...,Abstract_x000D_\nThis research use quantitativ...,"Dra. Hj. S Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST,SE,.MT",Manajemen
...,...,...,...,...,...,...,...,...
1026,160211100071,Husnul Hotimah,Analisis Cost Volume Profit Untuk Menentukan T...,ABSTRAK\nPenelitian ini bertujuan untuk menget...,ABSTRACT\nThis study aims to determine the cal...,"Hj. Evaliati Amaniyah, S.E., M.S.M.",NaN,Manajemen
1027,160211100291,Uswatun Hasanah,Pengaruh Pelatihan Dan Kompensasi Terhadap Pro...,"ABSTRAK\nUswatun Hasanah, 160211100291, Pengar...","ABSTRACT\nUswatun Hasanah, 160211100291, The E...","Dr. Raden Mas Mochammad Wispandono S.E ., MS",NaN,Manajemen
1028,160211100064,ACH FATHONI,PERAN SERVICE PERFORMANCE DAN CLIMATE ORGANIZA...,ABSTRAK\nTujuan dari penelitian ini adalah unt...,ABSTRACK\n The purpose of this study...,"YUDHI PRASETYA MADA, S.E., M.M.",NaN,Manajemen
1029,160211100030,INTAN YULLIA NINGSIH,BAURAN PROMOSI PADA DEALER YAMAHA TRETAN MOTOR...,ABSTRAK\nPenelitian ini bertujuan: (1) Untuk m...,ABSTRACK\nThis study aims: (1) To find out whe...,"DR. MOHAMMAD ARIEF, S.E., M.M.",NaN,Manajemen


In [ ]:
# Fungsi cleaning dengan emoji
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace('_x000D_', ' ')  # hapus karakter Excel khusus
    text = text.replace('\n', ' ').replace('\r', ' ')  # hapus line break
    text = text.lower()
    text = emoji.demojize(text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\W', ' ', text)
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Baca Excel
pta_df = pd.read_excel("pta.xlsx", engine='openpyxl', dtype=str).fillna("")

# Terapkan langsung ke kolom target
pta_df["abstrak_id_clean"] = pta_df["abstrak_id"].apply(clean_text)

In [ ]:
print("\nPTA (abstrak_id):")
pta_df[["abstrak_id", "abstrak_id_clean"]].head(5)


PTA (abstrak_id):


,abstrak_id,abstrak_id_clean
0,"ABSTRAK_x000D_\nSatiyah, Pengaruh Faktor-fakto...",abstrak satiyah pengaruh faktorfaktor pelatiha...
1,Tujuan penelitian ini adalah untuk mengetahui ...,tujuan penelitian ini adalah untuk mengetahui ...
2,,
3,Aplikasi nyata pemanfaatan teknologi informasi...,aplikasi nyata pemanfaatan teknologi informasi...
4,Abstrak_x000D_\nPenelitian ini menggunakan met...,abstrak penelitian ini menggunakan metode kuan...


In [ ]:
# Tokenisasi untuk PTA
pta_df["abstrak_id_tokens"] = pta_df["abstrak_id_clean"].apply(word_tokenize)

In [ ]:
print("\nPTA (abstrak_id_tokens):")
pta_df[["abstrak_id_clean", "abstrak_id_tokens"]].head(5)


PTA (abstrak_id_tokens):


,abstrak_id_clean,abstrak_id_tokens
0,abstrak satiyah pengaruh faktorfaktor pelatiha...,"[abstrak, satiyah, pengaruh, faktorfaktor, pel..."
1,tujuan penelitian ini adalah untuk mengetahui ...,"[tujuan, penelitian, ini, adalah, untuk, menge..."
2,,[]
3,aplikasi nyata pemanfaatan teknologi informasi...,"[aplikasi, nyata, pemanfaatan, teknologi, info..."
4,abstrak penelitian ini menggunakan metode kuan...,"[abstrak, penelitian, ini, menggunakan, metode..."


In [ ]:
# Stopwords untuk bahasa Indonesia
stop_words_id = set(stopwords.words('indonesian'))

# Filter stopwords di PTA
pta_df["abstrak_id_filtered"] = pta_df["abstrak_id_tokens"].apply(
    lambda tokens: [word for word in tokens if word not in stop_words_id]
)

In [ ]:
print("\nPTA (abstrak_id_filtered):")
pta_df[["abstrak_id_tokens", "abstrak_id_filtered"]].head(5)


PTA (abstrak_id_filtered):


,abstrak_id_tokens,abstrak_id_filtered
0,"[abstrak, satiyah, pengaruh, faktorfaktor, pel...","[abstrak, satiyah, pengaruh, faktorfaktor, pel..."
1,"[tujuan, penelitian, ini, adalah, untuk, menge...","[tujuan, penelitian, persepsi, brand, associat..."
2,[],[]
3,"[aplikasi, nyata, pemanfaatan, teknologi, info...","[aplikasi, nyata, pemanfaatan, teknologi, info..."
4,"[abstrak, penelitian, ini, menggunakan, metode...","[abstrak, penelitian, metode, kuantitatif, men..."


In [ ]:
factory = StemmerFactory()
indo_stemmer = factory.create_stemmer()

pta_df["abstrak_id_stemmed"] = pta_df["abstrak_id_filtered"].apply(
    lambda tokens: [indo_stemmer.stem(word) for word in tokens]
)

In [ ]:
print("\nPTA - Stemming & Lemmatization (abstrak_id):")
pta_df[["abstrak_id_filtered", "abstrak_id_stemmed"]].head(5)


PTA - Stemming & Lemmatization (abstrak_id):


,abstrak_id_filtered,abstrak_id_stemmed
0,"[abstrak, satiyah, pengaruh, faktorfaktor, pel...","[abstrak, satiyah, pengaruh, faktorfaktor, lat..."
1,"[tujuan, penelitian, persepsi, brand, associat...","[tuju, teliti, persepsi, brand, association, l..."
2,[],[]
3,"[aplikasi, nyata, pemanfaatan, teknologi, info...","[aplikasi, nyata, manfaat, teknologi, informas..."
4,"[abstrak, penelitian, metode, kuantitatif, men...","[abstrak, teliti, metode, kuantitatif, tekan, ..."


In [ ]:
# Fungsi expand kontraksi bahasa Indonesia
def expand_indonesian_contractions(text):
  contractions_dict = {
      "gak": "tidak", "ga": "tidak", "nggak": "tidak", "enggak": "tidak", "ngga": "tidak", "gk": "tidak", "tdk": "tidak", "tk": "tidak",
      "gue": "saya", "gw": "saya", "gua": "saya", "sy": "saya", "aq": "saya", "q": "saya", "ane": "saya",
      "lu": "kamu", "loe": "kamu", "lo": "kamu", "km": "kamu", "kmu": "kamu", "elu": "kamu",
      "dah": "sudah", "udah": "sudah", "sdh": "sudah", "udh": "sudah",
      "blm": "belum", "td": "tadi", "ntar": "nanti", "skr": "sekarang", "skrg": "sekarang", "skg": "sekarang",
      "kmrn": "kemarin", "kemrn": "kemarin", "kmarin": "kemarin",
      "aja": "saja", "aj": "saja", "sj": "saja",
      "nih": "ini", "nie": "ini", "ni": "ini", "tuh": "itu", "gtu": "begitu", "gitu": "begitu",
      "trs": "terus", "trus": "terus",
      "yg": "yang", "utk": "untuk", "dlm": "dalam", "dr": "dari", "dg": "dengan", "jd": "jadi", "jg": "juga",
      "krn": "karena", "tp": "tetapi", "tpi": "tetapi", "sm": "sama", "thd": "terhadap",
      "dll": "dan lain-lain", "dsb": "dan sebagainya", "dst": "dan seterusnya",
      "banget": "sekali", "bgt": "sekali", "sgt": "sangat", "sngt": "sangat",
      "lg": "sedang", "sdg": "sedang",
      "dl": "dulu", "pls": "tolong", "tolongin": "tolong", "plis": "tolong",
      "wkwk": "tertawa", "wkwkwk": "tertawa", "hehe": "tertawa kecil", "hihi": "tertawa kecil",
      "btw": "ngomong-ngomong", "imo": "menurut saya", "imho": "menurut saya", "cmiiw": "koreksi jika saya salah",
      "idk": "saya tidak tahu", "jk": "hanya bercanda",
      "ok": "baik", "oke": "baik", "okey": "baik", "sip": "baik",
      "ciyus": "serius", "serem": "menyeramkan",
      "kl": "kalau", "klo": "kalau", "klu": "kalau",
      "spy": "supaya", "spya": "supaya",
      "bbrp": "beberapa", "tsb": "tersebut", "trsbt": "tersebut",
      "dpt": "dapat", "bs": "bisa", "bsa": "bisa",
      "stlh": "setelah", "sblm": "sebelum",

      "mnj": "manajemen", "man": "manajemen", "mgt": "management",
      "org": "organisasi", "org2": "organisasi", "orgzt": "organisasi",
      "str": "struktur", "stkt": "struktur",
      "ldr": "leader", "ldrshp": "leadership", "pimp": "pimpinan", "pemimp": "pemimpin",
      "pln": "perencanaan", "renc": "perencanaan", "plan": "perencanaan",
      "orgz": "organizing", "orgzn": "organisasi",
      "dir": "directing", "pgn": "pengarahan",
      "cnt": "control", "cont": "control", "ctrl": "kontrol", "pengend": "pengendalian",
      "eff": "efisiensi", "effct": "efektivitas",
      "sdm": "sumber daya manusia", "hr": "human resource", "hrd": "human resource development",
      "res": "resource", "rsc": "resource", "sda": "sumber daya alam",
      "inv": "investasi", "invt": "investasi",
      "pmas": "pemasaran", "mkt": "marketing", "mktg": "marketing",
      "prod": "produksi", "prdks": "produksi",
      "fin": "finance", "keu": "keuangan", "akut": "akuntansi", "acct": "akuntansi",
      "ris": "risiko", "rsko": "risiko",
      "anal": "analisis", "eval": "evaluasi",
      "strtg": "strategi", "stg": "strategi",
      "ops": "operasi", "opr": "operasional", "oprs": "operasional",
      "bsc": "balanced scorecard", "swot": "analisis swot", "pest": "analisis pest",
      "csr": "corporate social responsibility", "gcn": "good corporate governance",
      "qm": "quality management", "iso": "standar iso",
      "kpi": "key performance indicator", "indik": "indikator",
      "knowl": "knowledge management", "kmgt": "knowledge management",
      "chg": "change management", "innv": "inovasi",
      "cnfl": "konflik", "cnflt": "konflik",
      "krj": "kerja", "tm": "tim", "tmwrk": "kerja sama tim",
      "cst": "cost", "faktorfaktor": "faktor", "dampakdampak": "dampak", "bya": "biaya",
      "val": "nilai", "valu": "value",
      "proj": "proyek", "prjk": "proyek",

      "med": "medis", "obat2": "obat-obatan", "rs": "rumah sakit",
      "dok": "dokter", "drg": "dokter gigi", "prof": "profesor",
      "pt": "perguruan tinggi", "univ": "universitas", "fak": "fakultas",
      "skripsi": "skripsi", "tesis": "tesis", "disertasi": "disertasi",
      "mhs": "mahasiswa", "mhsw": "mahasiswa"
  }


  pattern = r'\b(' + '|'.join(re.escape(key) for key in contractions_dict.keys()) + r')\b'

  def replace_match(match):
      return contractions_dict[match.group(0).lower()]

  expanded_text = re.sub(pattern, replace_match, text, flags=re.IGNORECASE)
  return expanded_text


pta_df["abstrak_id_expanded"] = pta_df["abstrak_id_stemmed"].apply(
    lambda tokens: expand_indonesian_contractions(" ".join(tokens)).split()
)


In [ ]:
print("\nPTA - Abstrak ID (expanded):")
pta_df[["abstrak_id_stemmed", "abstrak_id_expanded"]].head(5)


PTA - Abstrak ID (expanded):


,abstrak_id_stemmed,abstrak_id_expanded
0,"[abstrak, satiyah, pengaruh, faktorfaktor, lat...","[abstrak, satiyah, pengaruh, faktor, latih, ke..."
1,"[tuju, teliti, persepsi, brand, association, l...","[tuju, teliti, persepsi, brand, association, l..."
2,[],[]
3,"[aplikasi, nyata, manfaat, teknologi, informas...","[aplikasi, nyata, manfaat, teknologi, informas..."
4,"[abstrak, teliti, metode, kuantitatif, tekan, ...","[abstrak, teliti, metode, kuantitatif, tekan, ..."


In [ ]:
# Inisialisasi SpellChecker kosong
spell = SpellChecker(language=None)

# Load kamus Indonesia dari file
with open("00-indonesian-wordlist.lst", "r", encoding="latin-1") as f:
    indo_words = [line.strip() for line in f.readlines()]

spell.word_frequency.load_words(indo_words)

# Fungsi untuk koreksi kata
def correct_word(word):
    corr = spell.correction(word)
    return corr if corr is not None else word

# Terapkan spellcheck ke setiap baris dengan progress bar
corrected_texts = []
for tokens in tqdm(pta_df["abstrak_id_expanded"], desc="Spellchecking", unit="row"):
    corrected = [correct_word(word) for word in tokens]
    corrected_texts.append(corrected)

pta_df["abstrak_id_spellchecked"] = corrected_texts

Spellchecking: 100%|██████████| 1031/1031 [54:40<00:00,  3.18s/row]


In [ ]:
print("\nPTA - Abstrak ID (Cek Ejaan):")
pta_df[["abstrak_id_expanded", "abstrak_id_spellchecked"]].head(5)


PTA - Abstrak ID (Cek Ejaan):


,abstrak_id_expanded,abstrak_id_spellchecked
0,"[abstrak, satiyah, pengaruh, faktor, latih, ke...","[abstrak, aliyah, pengaruh, faktor, latih, kem..."
1,"[tuju, teliti, persepsi, brand, association, l...","[tuju, teliti, persepsi, band, association, la..."
2,[],[]
3,"[aplikasi, nyata, manfaat, teknologi, informas...","[aplikasi, nyata, manfaat, teknologi, informas..."
4,"[abstrak, teliti, metode, kuantitatif, tekan, ...","[abstrak, teliti, metode, kuantitatif, tekan, ..."


In [ ]:
# Gabungkan semua token jadi satu list besar
all_tokens = [word for tokens in pta_df["abstrak_id_spellchecked"] for word in tokens]

# Hitung frekuensi kata
word_freq = Counter(all_tokens)

# Ambil 20 kata paling sering muncul
common_words = word_freq.most_common(20)

print("=== 20 Kata Terbanyak ===")
for word, freq in common_words:
    print(f"{word} : {freq}")

# Informasi tambahan
total_words = len(all_tokens)                       # jumlah total kata
unique_words = len(word_freq)                       # jumlah kata unik
avg_words_per_data = np.mean([len(tokens) for tokens in pta_df["abstrak_id_spellchecked"]])  # rata-rata jumlah kata per data

print("\n=== Statistik Tambahan ===")
print(f"Jumlah total kata       : {total_words}")
print(f"Jumlah kata unik        : {unique_words}")
print(f"Rata-rata kata per data : {avg_words_per_data:.2f} kata")


=== 20 Kata Terbanyak ===
pengaruh : 5560
kerja : 5394
teliti : 4385
variabel : 3677
usaha : 2538
signifikan : 2494
uji : 2370
karyawan : 2273
nilai : 1912
hasil : 1788
analisis : 1500
positif : 1325
sampel : 1176
data : 1065
putus : 1040
tuju : 1024
parsial : 1003
metode : 968
simultan : 961
tingkat : 931

=== Statistik Tambahan ===
Jumlah total kata       : 149477
Jumlah kata unik        : 5502
Rata-rata kata per data : 144.98 kata


In [ ]:
# Pilih kolom yang ingin disimpan
cols_to_save = [
    "abstrak_id",
    "abstrak_id_clean",
    "abstrak_id_tokens",
    "abstrak_id_filtered",
    "abstrak_id_stemmed",
    "abstrak_id_expanded",
    "abstrak_id_spellchecked"
]

# Simpan ke Excel
pta_df[cols_to_save].to_excel("pta_processed.xlsx", index=False, engine='openpyxl')

print("✅ File berhasil disimpan sebagai pta_processed.xlsx")

✅ File berhasil disimpan sebagai pta_processed.xlsx


In [ ]:
pta_df_process = pd.read_excel("pta_processed.xlsx", engine='openpyxl')
pta_df_process

,abstrak_id,abstrak_id_clean,abstrak_id_tokens,abstrak_id_filtered,abstrak_id_stemmed,abstrak_id_expanded,abstrak_id_spellchecked
0,"ABSTRAK_x000D_\nSatiyah, Pengaruh Faktor-fakto...",abstrak satiyah pengaruh faktorfaktor pelatiha...,"['abstrak', 'satiyah', 'pengaruh', 'faktorfakt...","['abstrak', 'satiyah', 'pengaruh', 'faktorfakt...","['abstrak', 'satiyah', 'pengaruh', 'faktorfakt...","['abstrak', 'satiyah', 'pengaruh', 'faktor', '...","['abstrak', 'aliyah', 'pengaruh', 'faktor', 'l..."
1,Tujuan penelitian ini adalah untuk mengetahui ...,tujuan penelitian ini adalah untuk mengetahui ...,"['tujuan', 'penelitian', 'ini', 'adalah', 'unt...","['tujuan', 'penelitian', 'persepsi', 'brand', ...","['tuju', 'teliti', 'persepsi', 'brand', 'assoc...","['tuju', 'teliti', 'persepsi', 'brand', 'assoc...","['tuju', 'teliti', 'persepsi', 'band', 'associ..."
2,NaN,NaN,[],[],[],[],[]
3,Aplikasi nyata pemanfaatan teknologi informasi...,aplikasi nyata pemanfaatan teknologi informasi...,"['aplikasi', 'nyata', 'pemanfaatan', 'teknolog...","['aplikasi', 'nyata', 'pemanfaatan', 'teknolog...","['aplikasi', 'nyata', 'manfaat', 'teknologi', ...","['aplikasi', 'nyata', 'manfaat', 'teknologi', ...","['aplikasi', 'nyata', 'manfaat', 'teknologi', ..."
4,Abstrak_x000D_\nPenelitian ini menggunakan met...,abstrak penelitian ini menggunakan metode kuan...,"['abstrak', 'penelitian', 'ini', 'menggunakan'...","['abstrak', 'penelitian', 'metode', 'kuantitat...","['abstrak', 'teliti', 'metode', 'kuantitatif',...","['abstrak', 'teliti', 'metode', 'kuantitatif',...","['abstrak', 'teliti', 'metode', 'kuantitatif',..."
...,...,...,...,...,...,...,...
1026,ABSTRAK\nPenelitian ini bertujuan untuk menget...,abstrak penelitian ini bertujuan untuk mengeta...,"['abstrak', 'penelitian', 'ini', 'bertujuan', ...","['abstrak', 'penelitian', 'bertujuan', 'perhit...","['abstrak', 'teliti', 'tuju', 'hitung', 'tingk...","['abstrak', 'teliti', 'tuju', 'hitung', 'tingk...","['abstrak', 'teliti', 'tuju', 'hitung', 'tingk..."
1027,"ABSTRAK\nUswatun Hasanah, 160211100291, Pengar...",abstrak uswatun hasanah pengaruh pelatihan dan...,"['abstrak', 'uswatun', 'hasanah', 'pengaruh', ...","['abstrak', 'uswatun', 'hasanah', 'pengaruh', ...","['abstrak', 'uswatun', 'hasanah', 'pengaruh', ...","['abstrak', 'uswatun', 'hasanah', 'pengaruh', ...","['abstrak', 'uswatun', 'hadanah', 'pengaruh', ..."
1028,ABSTRAK\nTujuan dari penelitian ini adalah unt...,abstrak tujuan dari penelitian ini adalah untu...,"['abstrak', 'tujuan', 'dari', 'penelitian', 'i...","['abstrak', 'tujuan', 'penelitian', 'peran', '...","['abstrak', 'tuju', 'teliti', 'peran', 'servic...","['abstrak', 'tuju', 'teliti', 'peran', 'servic...","['abstrak', 'tuju', 'teliti', 'peran', 'servis..."
1029,ABSTRAK\nPenelitian ini bertujuan: (1) Untuk m...,abstrak penelitian ini bertujuan untuk mengeta...,"['abstrak', 'penelitian', 'ini', 'bertujuan', ...","['abstrak', 'penelitian', 'bertujuan', 'bauran...","['abstrak', 'teliti', 'tuju', 'baur', 'promosi...","['abstrak', 'teliti', 'tuju', 'baur', 'promosi...","['abstrak', 'teliti', 'tuju', 'baur', 'promosi..."


In [ ]:
# Simpan semua kata + frekuensi ke Excel
word_freq_df = pd.DataFrame(word_freq.items(), columns=["kata", "jumlah"])
word_freq_df = word_freq_df.sort_values(by="jumlah", ascending=False).reset_index(drop=True)
word_freq_df.to_excel("pta_word_frequency.xlsx", index=False, engine="openpyxl")

print("\n✅ File berhasil disimpan sebagai pta_word_frequency.xlsx.")


✅ File berhasil disimpan sebagai pta_word_frequency.xlsx.


In [ ]:
pta_df_wfq = pd.read_excel("pta_word_frequency.xlsx", engine='openpyxl')
pta_df_wfq.head(10)

,kata,jumlah
0,pengaruh,5560
1,kerja,5394
2,teliti,4385
3,variabel,3677
4,usaha,2538
5,signifikan,2494
6,uji,2370
7,karyawan,2273
8,nilai,1912
9,hasil,1788
